# Lab 5 - Bozulmayi Yakalayan Minik Test

**Kanitladigi tez:** Olcmezseniz bozulmayi goremezsiniz.

LLM ciktisi akici oldugu icin 'test edilmis' hissi verir.
Akicilik dogruluk demek degildir. Bu labda 15 dakikada kurulabilen
en kucuk degerlendirme (eval) duzenegini kuruyoruz.

## Kurulum

Asagidaki iki hucreyi sirayla calistirin. Bilgisayariniza hicbir sey
kurulmuyor: her sey sizin Colab calisma zamaninizda calisir ve
oturum kapaninca silinir.

In [ ]:
# 1/2 - Repoyu indir
!git clone -q https://github.com/silexi/guvenli-ai-mimarileri-lab.git 2>/dev/null || echo 'repo zaten var'
%cd -q guvenli-ai-mimarileri-lab
!pip install -q -U transformers accelerate 2>/dev/null
print('kurulum tamam')

In [ ]:
# 2/2 - Modeli sec ve yukle
import os, sys
sys.path.insert(0, '.')

# Uc secenek:
#   'colab' -> kendi calisma zamaninizda kucuk bir model (varsayilan)
#   'mock'  -> model yuklemeden, kayitli cevaplarla (yedek yol)
os.environ['LAB_SAGLAYICI'] = 'colab'

from ortak import llm
print(llm.durum())

# Modeli simdi yukleyelim ki sonraki hucreler beklemesin.
# GPU yoksa bu adim birkac dakika surebilir.
try:
    llm.model_yukle()
except Exception as hata:
    print('Model yuklenemedi:', hata)
    print('MOCK moda geciliyor, lab yapisi aynen calisacak.')
    os.environ['LAB_SAGLAYICI'] = 'mock'

print('\nLab 5 icin hazir.')

---
## 1. Altin set (golden set)

Bir eval'in tamami sudur: ornek girdiler + beklenen ciktilar + bir kontrol.
Bes ornekle baslamak, hic olcmemekten sonsuz kez iyidir.

In [ ]:
ALTIN_SET = [
    {'girdi': 'Faturam iki kez kesildi, acilen cozulsun.',
     'beklenen': {'kategori': 'faturalama', 'aciliyet': 'yuksek'}},
    {'girdi': 'Odemem gecti ama hesabima yansimadi.',
     'beklenen': {'kategori': 'odeme', 'aciliyet': 'orta'}},
    {'girdi': 'Panele giris yaparken hata aliyorum.',
     'beklenen': {'kategori': 'teknik', 'aciliyet': 'dusuk'}},
    {'girdi': 'Gecen ayki faturamda bir kalem anlamadim.',
     'beklenen': {'kategori': 'faturalama', 'aciliyet': 'orta'}},
    {'girdi': 'Sozlesmemi hemen iptal etmek istiyorum.',
     'beklenen': {'kategori': 'iptal', 'aciliyet': 'yuksek'}},
]

print(f'{len(ALTIN_SET)} ornek hazir.')

---
## 2. Degerlendirmeyi calistiralim

In [ ]:
from ortak import llm

PROMPT_V1 = '''Su destek talebini siniflandir.
Yalnizca JSON dondur: {"kategori": ..., "aciliyet": ...}
kategori: faturalama | odeme | teknik | iptal
aciliyet: dusuk | orta | yuksek

Talep: {talep}'''

def degerlendir(prompt_sablonu, senaryo, ad):
    gecen, hatalar = 0, []
    for i, ornek in enumerate(ALTIN_SET):
        istek = prompt_sablonu.replace('{talep}', ornek['girdi'])
        cevap = llm.sor(istek, sicaklik=0.0, senaryo=senaryo)
        cikti = llm.json_ayikla(cevap.metin) or {}

        uyum = all(cikti.get(k) == v
                   for k, v in ornek['beklenen'].items())
        if uyum:
            gecen += 1
        else:
            hatalar.append({
                'no': i + 1,
                'girdi': ornek['girdi'][:45],
                'beklenen': ornek['beklenen'],
                'gelen': cikti,
            })

    oran = gecen / len(ALTIN_SET)
    print(f'{ad}: {gecen}/{len(ALTIN_SET)} gecti  ({oran:.0%})')
    for h in hatalar:
        print(f"  [{h['no']}] {h['girdi']}")
        print(f"      beklenen: {h['beklenen']}")
        print(f"      gelen   : {h['gelen']}")
    return oran

temel = degerlendir(PROMPT_V1, 'lab5_v1', 'Surum 1 (temel)')

---
## 3. Prompt'u degistirelim

Simdi prompt'a masum gorunen bir degisiklik yapiyoruz. Boyle bir
degisiklik gercek hayatta 'daha iyi olsun diye' yapilir ve
olcum olmadan etkisi asla gorulmez.

In [ ]:
PROMPT_V2 = '''Su destek talebini siniflandir.
Yalnizca JSON dondur: {"kategori": ..., "aciliyet": ...}
kategori: faturalama | odeme | teknik | iptal
aciliyet: dusuk | orta | yuksek

Aciliyet belirlerken temkinli ol, gereksiz yere yuksek verme.

Talep: {talep}'''

yeni = degerlendir(PROMPT_V2, 'lab5_v2', 'Surum 2 (degistirilmis)')

In [ ]:
print()
print('=' * 55)
if yeni < temel:
    print(f'REGRESYON: {temel:.0%} -> {yeni:.0%}')
    print('Bu degisiklik uretime CIKMAMALI.')
elif yeni > temel:
    print(f'IYILESME: {temel:.0%} -> {yeni:.0%}')
else:
    print(f'DEGISIM YOK: {temel:.0%}')
print('=' * 55)
print()
print('Bu kontrol olmasaydi, degisiklik sessizce uretime giderdi')
print('ve etkisi ancak musteri sikayetiyle ortaya cikardi.')

> **1. gunde konustugumuz DoorDash ornegini hatirlayin.** Yayinladiklari
> mimaride her dagitim oncesi yuzlerce sentetik konusma calistirilip
> regresyon yakalaniyor. Buradaki bes ornek onun en kucuk halidir.
> Onemli olan sayinin buyuklugu degil, kontrolun VAR olmasi.

---
## 4. Bir sonraki adim: bunu CI'a baglamak

Bu defteri bir Python dosyasina cevirip prompt deposuna koyarsaniz,
her prompt degisikliginde otomatik calisir. Asagidaki iskelet yeterli.

In [ ]:
ISKELET = '''
# eval.py -- her prompt degisikliginde CI'da calisir

def test_siniflandirma_regresyonu():
    oran = degerlendir(AKTIF_PROMPT)
    assert oran >= 0.80, f"Regresyon: basari orani {oran:.0%}"
'''
print(ISKELET)

---
## Egzersiz

1. Altin sete kendi kurumunuzdan bes ornek daha ekleyin.
2. Esigi 0.80 yerine 1.00 yapin. Gercekci mi?
3. `aciliyet` alani icin ayri bir basari orani hesaplayin.
   Hangi alan daha kirilgan?
4. Bir de 'sema uyumu' metrigi ekleyin: cikti JSON olarak
   ayristirilabildi mi? Bu, dogrulugun onkosuludur.

---
## Iki gunun kapanisi

> Yapay zekanin basarisi modelde degil, modelin etrafina kurdugun
> mimaride belirlenir.

Bu repo acik kaliyor. Defterleri kendi verinizle calistirin.